PLOT MULTIPLE CONDITIONS

In [ ]:
# Histogram of track lifetimes

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys 
import os
import seaborn as sns
import matplotlib.ticker as ticker

pythonPackagePath = os.path.abspath(r'D:\LLSM-CME-ANALYSIS\Final\src')
sys.path.append(pythonPackagePath)

from intensity_time_plots import filter_track_ids_by_length_ranges, random_track_ids
from intensity_time_plots import intensity_time_plot, createBufferForLifetimeCohort
from intensity_time_plots import createBufferForLifetimeCohort_normalized, cumulative_plots, cumulative_plots_ax

In [ ]:
###### Set the following variables #######


# Background intensity for each channel

background_channel_1 = 240
background_channel_2 = 150
background_channel_3 = 170

# framerate (in milliseconds)

framerate_msec = 2500

# protein names (optional)

channel1_name = 'Arp2/3-Halo'
channel2_name = 'Dynamin2-tagGFP2'
channel3_name = 'AP2-tagRFPt'

In [ ]:
value_to_plot = 'voxel_sum_adjusted'
# value_to_plot = 'voxel_sum'
# value_to_plot = 'peak_mean'
# value_to_plot = 'peak_max'

if value_to_plot == 'voxel_sum_adjusted':
    background_channel_1 = 0
    background_channel_2 = 0
    background_channel_3 = 0

# use this list to see all possible values to plot
# track_df.columns

In [ ]:
# filtered_tracks_control = pd.read_pickle(r'Z:\Abhi\LLSM_Analysis\controlOS_analysis\datasets\filtered_tracks_final.pkl')
filtered_tracks_control = pd.read_pickle(r'Z:\Abhi\LLSM_Analysis\controlOS_analysis\archive\filtered_tracks_final.pkl')
filtered_tracks_treatment = pd.read_pickle(r'Z:\Abhi\LLSM_Analysis\41_OS_analysis\datasets\filtered_tracks_final.pkl')

# filtered_tracks_control = pd.read_pickle(r'Z:\Abhi\LLSM_Analysis\controlOS_analysis\datasets\filtered_tracks_final.pkl')
# filtered_tracks_treatment = pd.read_pickle(r'Z:\Abhi\LLSM_Analysis\control_1_analysis\datasets\filtered_tracks_final.pkl')

In [ ]:
# filtered_tracks_control
label1 = 'Control'
label2 = 'OS'

# label1 = 'Control1'
# label2 = 'Control2'

# label1 = 'CK689 50uM'
# label2 = 'CK666 50uM'

# title = 'Control vs 4:1 OS 5-10min'
# title = 'Control1 vs Control2'
# title = 'CK689 50uM vs CK666 50uM'


In [ ]:
# region_to_plot = 'Apical'
# region_to_plot = 'Lateral'
# region_to_plot = 'Basal'
region_to_plot = 'All'

if region_to_plot == 'All':
    filtered_tracks_all_control = filtered_tracks_control.copy()
    filtered_tracks_all_treatment = filtered_tracks_treatment.copy()

if region_to_plot != 'All':
    filtered_tracks_all_control = filtered_tracks_control.copy()
    filtered_tracks_all_treatment = filtered_tracks_treatment.copy()
    filtered_tracks_all_control = filtered_tracks_all_control[filtered_tracks_all_control['membrane_region'] == region_to_plot]
    filtered_tracks_all_treatment = filtered_tracks_all_treatment[filtered_tracks_all_treatment['membrane_region'] == region_to_plot]

print('You are about to plot the following region: ', region_to_plot)

In [ ]:
# range of track lengths to plot in each cohort, in frames
custom_length_ranges = [[5, 10], [11,15], [16, 20], [21, 25], [26, 30], [31, 40], [41,90]]

In [ ]:
# by default you always plot channel 3 positive (fixing this)

# e.g. [True, True] to plot all three channels
# or [True, False] to plot channels 1 and 2

plot_all_channels = False
channels_to_plot = [True, True]
if plot_all_channels:
    tracks_channel_subset_control = filtered_tracks_all_control.copy()
    tracks_channel_subset_treatment = filtered_tracks_all_treatment.copy()
else:
    channels_to_plot = channels_to_plot

    tracks_channel_subset_control = filtered_tracks_all_control[(filtered_tracks_all_control['channel1_positive'] == channels_to_plot[0]) &
                                    (filtered_tracks_all_control['channel2_positive'] == channels_to_plot[1])]

    tracks_channel_subset_treatment = filtered_tracks_all_treatment[(filtered_tracks_all_treatment['channel1_positive'] == channels_to_plot[0]) &
                                        (filtered_tracks_all_treatment['channel2_positive'] == channels_to_plot[1])]
# else:
#     tracks_channel_subset_control = filtered_tracks_all_control[filtered_tracks_all_control['channel2_positive'] == True]
#     tracks_channel_subset_treatment = filtered_tracks_all_treatment[filtered_tracks_all_treatment['channel2_positive'] == True]

In [ ]:
# tracks_channel_subset_control.head()[['channel2_positive', 'channel1_positive', 'membrane_region']]

In [ ]:

# Plot histogram of all track lengths (lifetimes)

# 'all lifetimes' is a variable of all the track lengths, multipilied by time per track
all_lifetimes_control =  tracks_channel_subset_control['track_length']*framerate_msec/1000
all_lifetimes_treatment =  tracks_channel_subset_treatment['track_length']*framerate_msec/1000


# Plot histograms of all_lifetimes_control and all_lifetimes_os
# on the same plot with 50 bins, and different colors
bins = np.linspace(0, 300, 51)  # Define the bin edges
plt.hist(all_lifetimes_control, bins=bins, color = 'orange', edgecolor='black', alpha=0.5, label=label1)
plt.hist(all_lifetimes_treatment, bins=bins, color = 'blue', edgecolor='black', alpha=0.35, label=label2)

plt.xlabel('AP2 lifetime (s)', fontsize = 12, weight = 'bold')  # X-axis label
plt.ylabel('Frequency', fontsize = 12, weight = 'bold')  # Y-axis label
# plt.title('Histogram of AP2-TagRFP-T lifetimes', fontsize = 14, weight = 'bold')  # Title of the histogram
# plt.title('Histogram of AP2 lifetimes', fontsize = 14, weight = 'bold')  # Title of the histogram

# Add labels for plot_all_channels, channels_to_plot, and region_to_plot
# if plot_all_channels == True:
#     plt.text(0.5, 0.8, f'Plotting all channels: {plot_all_channels}', transform=plt.gca().transAxes, fontsize=12, weight='bold')
# if plot_all_channels == False:
#     plt.text(0.5, 0.8, f'Plotting channels: {channels_to_plot}', transform=plt.gca().transAxes, fontsize=12, weight='bold')
# plt.text(0.5, 0.75, f'Plotting region: {region_to_plot}', transform=plt.gca().transAxes, fontsize=12, weight='bold')

plt.legend(prop = {'weight':'bold'})
plt.show()

In [ ]:
# Cumulative plot of track lifetimes
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

def plot_cumulative_lifetime(dmso_df, ck666_df, conditions_list, title = None):
    plt.figure(figsize=(8, 7))
    alpha_values = {'orange': 0.5, 'blue': 0.35}

    # Define colors for conditions
    conditions = {conditions_list[0]: (dmso_df, 'orange'), conditions_list[1]: (ck666_df, 'blue')}
    
    for label, (df, color) in conditions.items():
        subset = np.sort(df['track_length'])
        cum_freq = np.arange(1, len(subset) + 1) / len(subset)
        alpha = alpha_values.get(color, 1.0)
        plt.plot(subset, cum_freq, label=label, color=color, linewidth=2)

    # Formatting
    plt.xlabel('AP2 lifetime (s)', fontsize=12, weight='bold')
    plt.ylabel('Cumulative frequency', fontsize=12, weight='bold')
    plt.xlim(0, 305)
    plt.ylim(0, 1.05)

    # Add horizontal dashed reference line at 0.5
    plt.axhline(0.5, linestyle="--", color="black")
    
    # Add shaded region (0-50s)
    # plt.axvspan(0, 50, color='brown', alpha=0.2)

    # Title and legend
    plt.title(title, fontsize=14, weight='bold')

    legend_handles = []
    for label, (df, color) in conditions.items():
        alpha = alpha_values.get(color, 1.0)
        patch = mpatches.Patch(color=color,alpha = alpha, label=label)
        legend_handles.append(patch)
    plt.legend(handles = legend_handles, fontsize=12, loc='upper left', bbox_to_anchor=(0.025, 1), prop = {'weight':'bold'})

    # Show plot
    # plt.show()
    return plt


all_lifetimes_control_df = pd.DataFrame(all_lifetimes_control)
all_lifetimes_treatment_df = pd.DataFrame(all_lifetimes_treatment)

# plt = plot_cumulative_lifetime(all_lifetimes_control_df, all_lifetimes_treatment_df, [label1, label2], title)
plt = plot_cumulative_lifetime(all_lifetimes_control_df, all_lifetimes_treatment_df, [label1, label2])
# if plot_all_channels == True:
#     plt.text(0.5, 0.8, f'Plotting all channels: {plot_all_channels}', transform=plt.gca().transAxes, fontsize=12, weight='bold')
# if plot_all_channels == False:
#     plt.text(0.5, 0.8, f'Plotting channels: {channels_to_plot}', transform=plt.gca().transAxes, fontsize=12, weight='bold')
# plt.text(0.5, 0.75, f'Plotting region: {region_to_plot}', transform=plt.gca().transAxes, fontsize=12, weight='bold')

plt.show()


In [ ]:
# Histogram of maximum background corrected ARPC3 intensity values, to determine cutoff for ARPC3-positive tracks
columns = ['C1_adjusted_voxel_sum_peak', 'C2_adjusted_voxel_sum_peak']
marker = ['ARPC3', 'DNM2']

# # Max AP2 intensity per track
# control_AP2 = tracks_channel_subset_control['C3_adjusted_voxel_sum'].reset_index(drop=True)

# max_control_AP2 = []
# for j in range(len(control_AP2)):
#     max_control_AP2.append(max(control_AP2[j]))  

for i in range(len(columns)):
    # Create a figure with 2 subplots side by side
    fig, ax1 = plt.subplots(1, 1, figsize=(10, 5))

    control =  tracks_channel_subset_control[columns[i]].reset_index(drop=True)
    

    max_control = []
    for j in range(len(control)):
        max_control.append(control[j])  # For every element in arpc3_control, get the max value



    # # Normalize to max AP2 intensity, optional
    # max_control = (np.array(max_control) / np.array(max_control_AP2)).tolist()
    


    # Generate 50 equally spaced bins between the min and max of max_arpc3_control and max_arpc3_OS
    # so I can plot the histograms of max_arpc3_control and max_arpc3_OS on the same plot
    # with the same binning
    bins = np.linspace(min(max_control), max(max_control), 200)


    # Plot histogram on the first subplot
    ax1.hist(max_control, bins=bins, color='orange', edgecolor='black', alpha=0.5, label=label1)

    # Calculate the data range
    # x_min = min(max_control)
    # x_max = max(max_control)
    x_min = 0
    x_max = 40000
    
    # Round minimum down and maximum up to nearest 10
    x_min_rounded = np.floor(x_min / 10) * 10
    x_max_rounded = np.ceil(x_max / 10) * 10
    
    # Create tick marks rounded to the nearest 10
    num_ticks =25  # Adjust this number for more or fewer ticks
    
    # Generate ticks rounded to the nearest 10
    x_ticks = np.linspace(x_min_rounded, x_max_rounded, num_ticks)
    x_ticks_rounded = np.round(x_ticks / 10) * 10  # Round to nearest 10
    
    ax1.set_xticks(x_ticks_rounded)
    ax1.set_xticklabels([f'{int(x)}' for x in x_ticks_rounded], rotation=45, fontsize=10)
    
    ax1.set_xlabel('Intensity', fontsize=12, weight='bold')
    ax1.set_ylabel('Frequency', fontsize=12, weight='bold')
    ax1.set_title('Histogram of max background corrected' + ' ' + marker[i] + ' ' +  'intensities', fontsize=14, weight='bold')

    # set x limit to 0 and 20000
    ax1.set_xlim(x_min_rounded, x_max_rounded)

    # Adjust layout
    plt.tight_layout()
    plt.show()

In [ ]:
# columns = ['C1_adjusted_voxel_sum', 'C2_adjusted_voxel_sum', 'C3_adjusted_voxel_sum']
# marker = ['ARPC3', 'DNM2', 'AP2']
columns = ['C1_adjusted_voxel_sum']
marker = ['ArpC3']

# Max AP2 intensity per track
control_AP2 = tracks_channel_subset_control['C3_adjusted_voxel_sum'].reset_index(drop=True)
treatment_AP2 = tracks_channel_subset_treatment['C3_adjusted_voxel_sum'].reset_index(drop=True)

max_control_AP2 = []
for j in range(len(control_AP2)):
    max_control_AP2.append(max(control_AP2[j]))  
max_treatment_AP2 = []
for j in range(len(treatment_AP2)):
    max_treatment_AP2.append(max(treatment_AP2[j])) 

for i in range(len(columns)):
    # Create a figure with 2 subplots side by side
    # fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    fig1, ax1 = plt.subplots(figsize=(6, 5.25))
    fig2, ax2 = plt.subplots(figsize=(6, 5.25))

    control =  tracks_channel_subset_control[columns[i]].reset_index(drop=True)
    treatment =  tracks_channel_subset_treatment[columns[i]].reset_index(drop=True)

    max_control = []
    for j in range(len(control)):
        max_control.append(max(control[j]))  # For every element in arpc3_control, get the max value

    max_treatment = []
    for j in range(len(treatment)):
        max_treatment.append(max(treatment[j]))  # For every element in arpc3_OS, get the max value

    # Normalize to max AP2 intensity, optional
    max_control = (np.array(max_control) / np.array(max_control_AP2)).tolist()
    max_treatment = (np.array(max_treatment) / np.array(max_treatment_AP2)).tolist()

    # if i == 0:
    #     print(max_control.median(), max_os.median())


    # Generate 50 equally spaced bins between the min and max of max_arpc3_control and max_arpc3_OS
    # so I can plot the histograms of max_arpc3_control and max_arpc3_OS on the same plot
    # with the same binning
    bins = np.linspace(min(min(max_control), min(max_treatment)), max(max(max_control), max(max_treatment)), 50)


    # Plot histogram on the first subplot
    ax1.hist(max_control, bins=bins, color='orange', edgecolor='black', alpha=0.5, label=label1)
    ax1.hist(max_treatment, bins=bins, color='blue', edgecolor='black', alpha=0.35, label=label2)
    ax1.set_xlabel('Normalized Intensity (A.U.)', fontsize=12, weight='bold')
    ax1.set_ylabel('Frequency', fontsize=12, weight='bold')
    # ax1.set_title('Histogram of max background corrected' + ' ' + marker[i] + ' ' +  'intensities', fontsize=14, weight='bold')


    # Add labels for settings on histogram
    # if plot_all_channels == True:
    #     ax1.text(0.5, 0.75, f'Plotting all channels: {plot_all_channels}', transform=ax1.transAxes, fontsize=12, weight='bold')
    # if plot_all_channels == False:
    #     ax1.text(0.5, 0.75, f'Plotting channels: {channels_to_plot}', transform=ax1.transAxes, fontsize=12, weight='bold')
    # ax1.text(0.6, 0.70, f'Plotting region: {region_to_plot}', transform=ax1.transAxes, fontsize=12, weight='bold')
    ax1.legend(prop={'weight': 'bold'}, loc ='upper right')
    # Set x limit of ax1 to [0, 6]
    # ax1.set_xlim(0, 6)

    # Plot boxplot on the second subplot
    boxplot_data = [max_control, max_treatment]
    bp = ax2.boxplot(boxplot_data, patch_artist=True)
    median_line = bp['medians'][0]
    median_y = median_line.get_ydata()[0]  # Y-value of the median
    ax2.hlines(y=median_y, xmin=1, xmax=2, color='grey', linewidth=0.5, linestyle='--')

    # Loop through each median line and place value beside it
    for j, median in enumerate(bp['medians']):
        xdata = median.get_xdata()
        ydata = median.get_ydata()
        x = xdata.mean()   # horizontal center of median line
        y = ydata[0]       # vertical location of median

        if j == 0:
            # Left box: place text to the left
            ax2.text(x - 0.1, y, f'{y:.2f}', ha='right', va='center', fontsize=10, weight='bold', color='black')
        elif j == 1:
            # Right box: place text to the right
            ax2.text(x + 0.1, y, f'{y:.2f}', ha='left', va='center', fontsize=10, weight='bold', color='black')


    

    # Customize boxplot colors to match histogram
    bp['boxes'][0].set(facecolor='orange', alpha=0.5)
    bp['boxes'][1].set(facecolor='blue', alpha=0.35)
    for k in range(2):
        bp['medians'][k].set(color='black', linewidth=1.5)

    # # Add labels for settings on boxplot
    # if plot_all_channels == True:
    #     ax2.text(0.5, 0.95, f'Plotting all channels: {plot_all_channels}', transform=ax2.transAxes, fontsize=12, weight='bold')
    # if plot_all_channels == False:
    #     ax2.text(0.5, 0.95, f'Plotting channels: {channels_to_plot}', transform=ax2.transAxes, fontsize=12, weight='bold')
    # ax2.text(0.6, 0.90, f'Plotting region: {region_to_plot}', transform=ax2.transAxes, fontsize=12, weight='bold')

    # Set boxplot labels and title
    ax2.set_xticklabels([label1, label2], fontsize=12, weight='bold')
    ax2.set_ylabel('Normalized maximum Arp2/3 intensity (A.U.)', fontsize=12, weight='bold')
    # ax2.set_ylim(0, 6)
    # ax2.set_title('Boxplot of max background corrected' + ' ' + marker[i] + ' ' +  'intensities', fontsize=14, weight='bold')

    # Adjust layout
    plt.tight_layout()
    plt.show()

In [ ]:
# For each track, looking at the position of the max intensity ARPC3 spot
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable

# Create a figure with 2 subplots side by side
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Extract the background corrected voxel sum for channel 1
control = tracks_channel_subset_control['C1_adjusted_voxel_sum'].reset_index(drop=True)
treatment = tracks_channel_subset_treatment['C1_adjusted_voxel_sum'].reset_index(drop=True)

# Extract position coordinates
control_x = tracks_channel_subset_control['mu_x'].reset_index(drop=True)
control_y = tracks_channel_subset_control['mu_y'].reset_index(drop=True)
treatment_x = tracks_channel_subset_treatment['mu_x'].reset_index(drop=True)
treatment_y = tracks_channel_subset_treatment['mu_y'].reset_index(drop=True)

# Lists to store max intensity and corresponding coordinates
max_control_intensity = []
max_control_x = []
max_control_y = []

max_treatment_intensity = []
max_treatment_x = []
max_treatment_y = []

# Find max intensity and corresponding coordinates for control
for j in range(len(control)):
    max_val = max(control[j])
    max_control_intensity.append(max_val)
    
    # Find the index of the maximum intensity frame in this track
    max_idx = list(control[j]).index(max_val)
    # print(j)
    # print(max_idx)
    
    # Get the x and y coordinates at that frame
    max_control_x.append(control_x[j].reset_index(drop = True)[max_idx])
    max_control_y.append(control_y[j].reset_index(drop = True)[max_idx])

# Find max intensity and corresponding coordinates for OS
for j in range(len(treatment)):
    max_val = max(treatment[j])
    max_treatment_intensity.append(max_val)
    
    # Find the index of the maximum intensity frame in this track
    max_idx = list(treatment[j]).index(max_val)
    
    # Get the x and y coordinates at that frame
    max_treatment_x.append(treatment_x[j].reset_index(drop = True)[max_idx])
    max_treatment_y.append(treatment_y[j].reset_index(drop = True)[max_idx])

# Create color normalization for the intensity values
vmin = min(min(max_control_intensity), min(max_treatment_intensity))
vmax = max(max(max_control_intensity), max(max_treatment_intensity))
norm = Normalize(vmin=vmin, vmax=vmax)

# Control scatter plot
scatter1 = ax1.scatter(max_control_x, max_control_y, c=max_control_intensity, 
                       cmap='plasma', alpha=0.7, s=50, edgecolors='black', linewidths=0.5, norm=norm)

# OS scatter plot
scatter2 = ax2.scatter(max_treatment_x, max_treatment_y, c=max_treatment_intensity, 
                       cmap='plasma', alpha=0.7, s=50, edgecolors='black', linewidths=0.5, norm=norm)

# Add color bar
cbar_ax = fig.add_axes([0.92, 0.15, 0.02, 0.7])  # [left, bottom, width, height]
cbar = fig.colorbar(ScalarMappable(norm=norm, cmap='plasma'), cax=cbar_ax)
cbar.ax.set_ylabel('Max ARPC3 Intensity', fontsize=12, weight='bold')
cbar.ax.tick_params(size=8, labelsize=10)

# Set titles and labels
ax1.set_title( label1 + ' ' + '- Max ARPC3 Intensity Positions', fontsize=14, weight='bold')
ax2.set_title( label2 + ' ' + '- Max ARPC3 Intensity Positions', fontsize=14, weight='bold')

for ax in [ax1, ax2]:
    # Move text to top-left corner with better spacing and transparency
    if 'plot_all_channels' in globals():
        if plot_all_channels == True:
            ax.text(0.02, 0.98, f'Plotting channels: {plot_all_channels}', 
                   transform=ax.transAxes, fontsize=10, weight='bold',
                   bbox=dict(facecolor='white', alpha=0.7), verticalalignment='top')
        else:
            ax.text(0.02, 0.98, f'Plotting channels: {channels_to_plot}', 
                   transform=ax.transAxes, fontsize=10, weight='bold',
                   bbox=dict(facecolor='white', alpha=0.7), verticalalignment='top')
    
    if 'region_to_plot' in globals():
        ax.text(0.02, 0.93, f'Plotting region: {region_to_plot}', 
               transform=ax.transAxes, fontsize=10, weight='bold',
               bbox=dict(facecolor='white', alpha=0.7), verticalalignment='top')

# Adjust layout
plt.subplots_adjust(left=0.05, right=0.9, bottom=0.1, top=0.9, wspace=0.2)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from mpl_toolkits.mplot3d import Axes3D  # Import for 3D plotting

# Create a figure with 2 subplots side by side for 3D plotting
fig = plt.figure(figsize=(15, 7))
ax1 = fig.add_subplot(121, projection='3d')  # 3D subplot for control
ax2 = fig.add_subplot(122, projection='3d')  # 3D subplot for OS

# Extract the background corrected voxel sum for channel 1
control = tracks_channel_subset_control['C1_adjusted_voxel_sum'].reset_index(drop=True)
treatment = tracks_channel_subset_treatment['C1_adjusted_voxel_sum'].reset_index(drop=True)

# Extract position coordinates
control_x = tracks_channel_subset_control['mu_x'].reset_index(drop=True)
control_y = tracks_channel_subset_control['mu_y'].reset_index(drop=True)
control_z = tracks_channel_subset_control['mu_z'].reset_index(drop=True)  # Added z-coordinate
treatment_x = tracks_channel_subset_treatment['mu_x'].reset_index(drop=True)
treatment_y = tracks_channel_subset_treatment['mu_y'].reset_index(drop=True)
treatment_z = tracks_channel_subset_treatment['mu_z'].reset_index(drop=True)  # Added z-coordinate

# Lists to store max intensity and corresponding coordinates
max_control_intensity = []
max_control_x = []
max_control_y = []
max_control_z = []  # Added z-coordinate list

max_treatment_intensity = []
max_treatment_x = []
max_treatment_y = []
max_treatment_z = []  # Added z-coordinate list

# Find max intensity and corresponding coordinates for control
for j in range(len(control)):
    max_val = max(control[j])
    max_control_intensity.append(max_val)
    
    # Find the index of the maximum intensity frame in this track
    max_idx = list(control[j]).index(max_val)
    
    # Get the x, y, and z coordinates at that frame
    max_control_x.append(control_x[j].reset_index(drop=True)[max_idx])
    max_control_y.append(control_y[j].reset_index(drop=True)[max_idx])
    max_control_z.append(control_z[j].reset_index(drop=True)[max_idx])  # Added z-coordinate

# Find max intensity and corresponding coordinates for OS
for j in range(len(treatment)):
    max_val = max(treatment[j])
    max_treatment_intensity.append(max_val)
    
    # Find the index of the maximum intensity frame in this track
    max_idx = list(treatment[j]).index(max_val)
    
    # Get the x, y, and z coordinates at that frame
    max_treatment_x.append(treatment_x[j].reset_index(drop=True)[max_idx])
    max_treatment_y.append(treatment_y[j].reset_index(drop=True)[max_idx])
    max_treatment_z.append(treatment_z[j].reset_index(drop=True)[max_idx])  # Added z-coordinate

# Create color normalization for the intensity values
vmin = min(min(max_control_intensity), min(max_treatment_intensity))
vmax = max(max(max_control_intensity), max(max_treatment_intensity))
norm = Normalize(vmin=vmin, vmax=vmax)

# Control scatter plot (3D)
scatter1 = ax1.scatter(max_control_x, max_control_y, max_control_z, 
                       c=max_control_intensity, cmap='plasma', alpha=0.7, 
                       s=50, edgecolors='black', linewidths=0.5, norm=norm)

# OS scatter plot (3D)
scatter2 = ax2.scatter(max_treatment_x, max_treatment_y, max_treatment_z, 
                       c=max_treatment_intensity, cmap='plasma', alpha=0.7, 
                       s=50, edgecolors='black', linewidths=0.5, norm=norm)

# Add color bar
cbar_ax = fig.add_axes([0.92, 0.15, 0.02, 0.7])  # [left, bottom, width, height]
cbar = fig.colorbar(ScalarMappable(norm=norm, cmap='plasma'), cax=cbar_ax)
cbar.ax.set_ylabel('Max ARPC3 Intensity', fontsize=12, weight='bold')
cbar.ax.tick_params(size=8, labelsize=10)

# Set titles and labels
ax1.set_title(label1 + ' ' + '- Max ARPC3 Intensity Positions', fontsize=14, weight='bold')
ax2.set_title(label2 + ' ' + '- Max ARPC3 Intensity Positions', fontsize=14, weight='bold')

# Set axis labels for 3D plots
for ax in [ax1, ax2]:
    ax.set_xlabel('X Position (pixels)', fontsize=12, weight='bold')
    ax.set_ylabel('Y Position (pixels)', fontsize=12, weight='bold')
    ax.set_zlabel('Z Position (pixels)', fontsize=12, weight='bold')  # Added z-axis label
    
    # Add view angle for better visualization
    ax.view_init(elev=30, azim=30)
    
    # Move text to top-left corner with better spacing and transparency
    if 'plot_all_channels' in globals():
        if plot_all_channels == True:
            ax.text2D(0.02, 0.98, f'Plotting channels: {plot_all_channels}', 
                     transform=ax.transAxes, fontsize=10, weight='bold',
                     bbox=dict(facecolor='white', alpha=0.7), verticalalignment='top')
        else:
            ax.text2D(0.02, 0.98, f'Plotting channels: {channels_to_plot}', 
                     transform=ax.transAxes, fontsize=10, weight='bold',
                     bbox=dict(facecolor='white', alpha=0.7), verticalalignment='top')
    
    if 'region_to_plot' in globals():
        ax.text2D(0.02, 0.93, f'Plotting region: {region_to_plot}', 
                 transform=ax.transAxes, fontsize=10, weight='bold',
                 bbox=dict(facecolor='white', alpha=0.7), verticalalignment='top')

# Adjust layout
plt.subplots_adjust(left=0.05, right=0.9, bottom=0.1, top=0.9, wspace=0.2)
plt.show()

In [ ]:
# Histogram of max ARPC3 intensities
# channel 1 is the ARPC3 channel, channel 2 is the Dynamin2 channel, and channel 3 is the AP2 channel
max_arpc3_control =  tracks_channel_subset_control['c1_peak']
max_arpc3_treatment =  tracks_channel_subset_treatment['c1_peak']

# Generate 50 equally spaced bins between the min and max of max_arpc3_control and max_arpc3_OS
# so I can plot the histograms of max_arpc3_control and max_arpc3_OS on the same plot
# with the same binning
bins = np.linspace(min(max_arpc3_control.min(), max_arpc3_treatment.min()), max(max_arpc3_control.max(), max_arpc3_treatment.max()), 50)

plt.hist(max_arpc3_control, bins = bins, color = 'orange', edgecolor='black', alpha=0.5, label=label1)
plt.hist(max_arpc3_treatment, bins = bins, color = 'blue', edgecolor='black', alpha=0.35, label=label2)

plt.xlabel('Intensity', fontsize = 12, weight = 'bold')  # X-axis label
plt.ylabel('Frequency', fontsize = 12, weight = 'bold')  # Y-axis label
plt.title('Histogram of max ARPC3 intensities', fontsize = 14, weight = 'bold')  # Title of the histogram
# Add labels for plot_all_channels, channels_to_plot, and region_to_plot
if plot_all_channels == True:
    plt.text(0.5, 0.8, f'Plotting all channels: {plot_all_channels}', transform=plt.gca().transAxes, fontsize=12, weight='bold')
if plot_all_channels == False:
    plt.text(0.5, 0.8, f'Plotting channels: {channels_to_plot}', transform=plt.gca().transAxes, fontsize=12, weight='bold')
plt.text(0.5, 0.75, f'Plotting region: {region_to_plot}', transform=plt.gca().transAxes, fontsize=12, weight='bold')

plt.legend(prop = {'weight':'bold'})
plt.show()

In [ ]:
# Histogram of max ARPC3 intensities with boxplot
# channel 1 is the ARPC3 channel, channel 2 is the Dynamin2 channel, and channel 3 is the AP2 channel
max_arpc3_control = tracks_channel_subset_control['c1_peak']
max_arpc3_treatment = tracks_channel_subset_treatment['c1_peak']

# Create a figure with 2 subplots side by side
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Generate 50 equally spaced bins between the min and max of max_arpc3_control and max_arpc3_OS
bins = np.linspace(min(max_arpc3_control.min(), max_arpc3_treatment.min()), max(max_arpc3_control.max(), max_arpc3_treatment.max()), 50)

# Plot histogram on the first subplot
ax1.hist(max_arpc3_control, bins=bins, color='orange', edgecolor='black', alpha=0.5, label=label1)
ax1.hist(max_arpc3_treatment, bins=bins, color='blue', edgecolor='black', alpha=0.35, label=label2)
ax1.set_xlabel('Intensity', fontsize=12, weight='bold')
ax1.set_ylabel('Frequency', fontsize=12, weight='bold')
ax1.set_title('Histogram of max ARPC3 intensities', fontsize=14, weight='bold')

# Add labels for settings on histogram
if plot_all_channels == True:
    ax1.text(0.5, 0.75, f'Plotting all channels: {plot_all_channels}', transform=ax1.transAxes, fontsize=12, weight='bold')
if plot_all_channels == False:
    ax1.text(0.5, 0.75, f'Plotting channels: {channels_to_plot}', transform=ax1.transAxes, fontsize=12, weight='bold')
ax1.text(0.6, 0.70, f'Plotting region: {region_to_plot}', transform=ax1.transAxes, fontsize=12, weight='bold')
ax1.legend(prop={'weight': 'bold'}, loc ='upper right')

# Plot boxplot on the second subplot
boxplot_data = [max_arpc3_control, max_arpc3_treatment]
bp = ax2.boxplot(boxplot_data, patch_artist=True)

# Customize boxplot colors to match histogram
bp['boxes'][0].set(facecolor='orange', alpha=0.5)
bp['boxes'][1].set(facecolor='blue', alpha=0.35)
for i in range(2):
    bp['medians'][i].set(color='black', linewidth=1.5)

# Add labels for settings on boxplot
if plot_all_channels == True:
    ax2.text(0.5, 0.95, f'Plotting all channels: {plot_all_channels}', transform=ax2.transAxes, fontsize=12, weight='bold')
if plot_all_channels == False:
    ax2.text(0.5, 0.95, f'Plotting channels: {channels_to_plot}', transform=ax2.transAxes, fontsize=12, weight='bold')
ax2.text(0.6, 0.90, f'Plotting region: {region_to_plot}', transform=ax2.transAxes, fontsize=12, weight='bold')

# Set boxplot labels and title
ax2.set_xticklabels([label1, label2], fontsize=12, weight='bold')
ax2.set_ylabel('Intensity', fontsize=12, weight='bold')
ax2.set_title('Boxplot of max ARPC3 intensities', fontsize=14, weight='bold')

# Adjust layout
plt.tight_layout()
plt.show()

In [ ]:
# Boxplot of max ARPC3 intensities, with all surfaces on the same plot
def plot_arpc3_surface_comparison(control_df, os_df, label1, label2, plot_title="ARPC3 Max Intensity by Surface Region"):
    """
    Create boxplots comparing ARPC3 max intensities between control and OS conditions
    across different surface regions (apical, lateral, basal).
    
    Parameters:
    -----------
    control_df : DataFrame
        The control condition data
    os_df : DataFrame
        The osmotic shock data
    plot_title : str
        Title for the plot
    """
    # Create figure
    fig, ax = plt.subplots(figsize=(12, 6))
    
    # Define regions and their positions on x-axis
    regions = ['Apical', 'Lateral', 'Basal']
    x_positions = np.arange(len(regions))
    width = 0.35  # Width of bars
    
    # Colors for conditions
    control_color = 'orange'
    os_color = 'blue'
    
    # Lists to store boxplot objects and their positions
    boxplots = []
    box_positions = []
    
    # Process each region
    for i, region in enumerate(regions):
        # Filter data for this region
        control_region = control_df[control_df['membrane_region'] == region]['c1_peak']
        os_region = os_df[os_df['membrane_region'] == region]['c1_peak']
        
        # Calculate positions for boxplots
        control_pos = i - width/2
        os_pos = i + width/2
        
        # Add positions to the list
        box_positions.extend([control_pos, os_pos])
        
        # Add data to the list
        boxplots.append(control_region)
        boxplots.append(os_region)
    
    # Create boxplots
    bp = ax.boxplot(boxplots, positions=box_positions, widths=width*0.8, patch_artist=True,)
    
    # Color the boxplots
    for i, box in enumerate(bp['boxes']):
        if i % 2 == 0:  # Control
            box.set(facecolor=control_color, alpha=0.6)
        else:  # OS
            box.set(facecolor=os_color, alpha=0.6)
    
    # Set x-axis ticks and labels
    ax.set_xticks(x_positions)
    ax.set_xticklabels(regions, fontweight='bold', fontsize=12)
    
    # Add sample size annotations
    for i, region in enumerate(regions):
        control_region = control_df[control_df['membrane_region'] == region]['c1_peak']
        os_region = os_df[os_df['membrane_region'] == region]['c1_peak']
        
        control_pos = i - width/2
        os_pos = i + width/2
        
        # Calculate median for positioning text
        control_median = control_region.median()
        os_median = os_region.median()    
    # Add legend
    control_patch = plt.Rectangle((0, 0), 1, 1, facecolor=control_color, alpha=0.6)
    os_patch = plt.Rectangle((0, 0), 1, 1, facecolor=os_color, alpha=0.6)
    ax.legend([control_patch, os_patch], [label1, label2], 
             loc='upper right', fontsize=12, frameon=True, fancybox=True, prop = {'weight':'bold'})
    
    # Set labels and title
    ax.set_ylabel('ARPC3 Max Intensity', fontsize=14, fontweight='bold')
    ax.set_title(plot_title, fontsize=16, fontweight='bold')
    
    # Set y-axis limit with some padding for significance markers
    upper_y = max([df[df['membrane_region'].isin(regions)]['c1_peak'].max() 
                  for df in [control_df, os_df]]) * 1.2
    ax.set_ylim(0, upper_y)
    
    plt.tight_layout()
    return fig

# by default you always plot channel 3 positive (fixing this)

# e.g. [True, True] to plot all three channels
# or [True, False] to plot channels 1 and 2

plot_all_channels = False
channels_to_plot = [True, True]
if plot_all_channels:
    tracks_control = filtered_tracks_control.copy()
    tracks_treatment = filtered_tracks_treatment.copy()
else:
    channels_to_plot = channels_to_plot

    tracks_control = filtered_tracks_control[(filtered_tracks_control['channel1_positive'] == channels_to_plot[0]) &
                                    (filtered_tracks_control['channel2_positive'] == channels_to_plot[1])]

    tracks_treatment = filtered_tracks_treatment[(filtered_tracks_treatment['channel1_positive'] == channels_to_plot[0]) &
                                        (filtered_tracks_treatment['channel2_positive'] == channels_to_plot[1])]


fig = plot_arpc3_surface_comparison(tracks_control, tracks_treatment, label1 = label1, label2 = label2)
plt.show()

In [ ]:
# Histogram of max AP2-TagRFP-T intensities
# channel 1 is the ARPC3 channel, channel 2 is the Dynamin2 channel, and channel 3 is the AP2 channel

In [ ]:
# AP2 lifetimes for ARPC3 negative and positive tracks
# channel 1 is the ARPC3 channel, channel 2 is the Dynamin2 channel, and channel 3 is the AP2 channel

to_plot = label1 # Be very careful with this label
df = filtered_tracks_all_control.copy()
# to_plot = label2 # Be very careful with this label
# df = filtered_tracks_all_treatment.copy()
# Histograms of AP2 lifetimes for ARPC3 negative and positive tracks
c1nc2p = df[(df['channel1_positive'] == False) &
                                    (df['channel2_positive'] == True)]

c1nc2p_lifetimes = c1nc2p['track_length']*framerate_msec/1000

c1pc2p = df[(df['channel1_positive'] == True) &
                                    (df['channel2_positive'] == True)]

c1pc2p_lifetimes = c1pc2p['track_length']*framerate_msec/1000

# Plot histograms of all_lifetimes_control and all_lifetimes_os
# on the same plot with 50 bins, and different colors
bins = np.linspace(0, 300, 51)  # Define the bin edges
# plt.hist(c1nc2p_lifetimes, bins=bins, color = 'orange', edgecolor='black', alpha=0.5, label='AP2+ DNM2+ ARPC3-')
# plt.hist(c1pc2p_lifetimes, bins=bins, color = 'blue', edgecolor='black', alpha=0.35, label='AP2+ DNM2+ ARPC3+')
plt.hist(c1nc2p_lifetimes, bins=bins, color = 'orange', edgecolor='black', density = True, alpha=0.5, label='AP2+ DNM2+ ARPC3-')
plt.hist(c1pc2p_lifetimes, bins=bins, color = 'blue', edgecolor='black', density = True, alpha=0.35, label='AP2+ DNM2+ ARPC3+')

plt.xlabel('Time (s)', fontsize = 12, weight = 'bold')  # X-axis label
plt.ylabel('Frequency', fontsize = 12, weight = 'bold')  # Y-axis label
plt.title('Histogram of AP2-TagRFP-T lifetimes', fontsize = 14, weight = 'bold')  # Title of the histogram
# Add labels for plot_all_channels, channels_to_plot, and region_to_plot
# if plot_all_channels == True:
#     plt.text(0.5, 0.8, f'Plotting all channels: {plot_all_channels}', transform=plt.gca().transAxes, fontsize=12, weight='bold')
# if plot_all_channels == False:
#     plt.text(0.5, 0.8, f'Plotting channels: {channels_to_plot}', transform=plt.gca().transAxes, fontsize=12, weight='bold')
plt.text(0.485, 0.8, f'Condition: {to_plot}', transform=plt.gca().transAxes, fontsize=12, weight='bold')
plt.text(0.55, 0.75, f'Plotting region: {region_to_plot}', transform=plt.gca().transAxes, fontsize=12, weight='bold')

plt.legend(prop = {'weight':'bold'})
plt.show()

In [ ]:
# # Set style parameters for better-looking plots
# plt.rcParams.update({'font.size': 12})
# sns.set_style("whitegrid")

# Function to plot the lifetime histograms for specified conditions
def plot_lifetime_by_region(control_df, os_df, label1, label2, region_name="All", normalize=True):
    """
    Plot lifetime histograms for tracks with AP2+ DNM2+ ARPC3- vs AP2+ DNM2+ ARPC3+
    
    Parameters:
    -----------
    control_df : DataFrame
        The control condition data
    os_df : DataFrame
        The osmotic shock data
    region_name : str
        The cell region to filter for (or "All")
    normalize : bool
        Whether to normalize the histograms (density=True)
    """
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Filter for region if specified
    if region_name != "All":
        control_subset = control_df[control_df['membrane_region'] == region_name].copy()
        os_subset = os_df[os_df['membrane_region'] == region_name].copy()
    else:
        control_subset = control_df.copy()
        os_subset = os_df.copy()
    
    # Convert track length to seconds
    control_subset['lifetime'] = control_subset['track_length'] * framerate_msec / 1000
    os_subset['lifetime'] = os_subset['track_length'] * framerate_msec / 1000
    
    # Define bins for consistency across plots
    bins = np.linspace(0, 300, 51)  # 0 to 160 seconds, 40 bins
    
    # Filter for AP2+ DNM2+ ARPC3+ and AP2+ DNM2+ ARPC3-
    for i, (df, title) in enumerate([(control_subset, f"{label1} - {region_name}"), 
                                     (os_subset, f"{label2} - {region_name}")]):
        # AP2+ is assumed for all tracks
        arpc3_pos = df[(df['channel1_positive'] == True) & (df['channel2_positive'] == True)]
        arpc3_neg = df[(df['channel1_positive'] == False) & (df['channel2_positive'] == True)]
        
        # Plot the histograms
        axes[i].hist(arpc3_neg['lifetime'], bins=bins, alpha=0.7, density=normalize,
                    color='skyblue', label='AP2+ DNM2+ ARPC3-')
        axes[i].hist(arpc3_pos['lifetime'], bins=bins, alpha=0.7, density=normalize,
                    color='salmon', label='AP2+ DNM2+ ARPC3+')
        
        # Add labels and legend
        axes[i].set_xlabel('CME lifetime (s)', fontweight='bold')
        if normalize:
            axes[i].set_ylabel('Frequency density', fontweight='bold')
        else:
            axes[i].set_ylabel('Count', fontweight='bold')
        axes[i].set_title(title, fontweight='bold')
        axes[i].legend()
        
        # Add track counts to the legend
        handles, labels = axes[i].get_legend_handles_labels()
        labels[0] += f" (n={len(arpc3_neg)})"
        labels[1] += f" (n={len(arpc3_pos)})"
        axes[i].legend(handles, labels)
    
    plt.tight_layout()
    return fig

# Create plots for each region
regions = ["All", "Apical", "Lateral", "Basal"]

# For each region, create and display the plot
for region in regions:
    fig = plot_lifetime_by_region(filtered_tracks_control, filtered_tracks_treatment, label1, label2, region_name=region)
    plt.show()


# Calculate statistics for a summary table
def calculate_region_stats(control_df, os_df, label1, label2):
    """Calculate summary statistics for each region and condition"""
    regions = ["All", "Apical", "Lateral", "Basal"]
    stats = []
    
    for region in regions:
        for df_name, df in [(label1, control_df), (label2, os_df)]:
            if region != "All":
                subset = df[df['membrane_region'] == region].copy()
            else:
                subset = df.copy()
                
            # Count AP2+ DNM2+ ARPC3+ and AP2+ DNM2+ ARPC3- tracks
            ap2_dnm2_arpc3_pos = subset[(subset['channel1_positive'] == True) & 
                                         (subset['channel2_positive'] == True)].copy()
            ap2_dnm2_arpc3_neg = subset[(subset['channel1_positive'] == False) & 
                                         (subset['channel2_positive'] == True)].copy()
            
            # Calculate mean lifetimes
            ap2_dnm2_arpc3_pos['lifetime'] = ap2_dnm2_arpc3_pos['track_length'] * framerate_msec / 1000
            ap2_dnm2_arpc3_neg['lifetime'] = ap2_dnm2_arpc3_neg['track_length'] * framerate_msec / 1000
            
            mean_pos = ap2_dnm2_arpc3_pos['lifetime'].mean() if len(ap2_dnm2_arpc3_pos) > 0 else 0
            mean_neg = ap2_dnm2_arpc3_neg['lifetime'].mean() if len(ap2_dnm2_arpc3_neg) > 0 else 0
            
            stats.append({
                "Region": region,
                "Condition": df_name,
                "AP2+ DNM2+ ARPC3+ Count": len(ap2_dnm2_arpc3_pos),
                "AP2+ DNM2+ ARPC3- Count": len(ap2_dnm2_arpc3_neg),
                "AP2+ DNM2+ ARPC3+ Mean Lifetime (s)": mean_pos,
                "AP2+ DNM2+ ARPC3- Mean Lifetime (s)": mean_neg
            })
    
    return pd.DataFrame(stats)

# Display summary statistics
stats_df = calculate_region_stats(filtered_tracks_control, filtered_tracks_treatment, label1, label2)
display(stats_df)

# Create a more visual plot of the mean lifetimes
plt.figure(figsize=(12, 8))

# Group by region and condition
grouped = stats_df.groupby(['Region', 'Condition'])

# For each region, plot the mean lifetimes for each condition and ARPC3 status
regions = ["All", "Apical", "Lateral", "Basal"]
pos = np.arange(len(regions))
width = 0.2  # Width of the bars

for i, cond in enumerate([label1, label2]):
    pos_means = [grouped.get_group((region, cond))["AP2+ DNM2+ ARPC3+ Mean Lifetime (s)"].values[0] 
                for region in regions]
    neg_means = [grouped.get_group((region, cond))["AP2+ DNM2+ ARPC3- Mean Lifetime (s)"].values[0] 
                for region in regions]
    
    plt.bar(pos + i*width - width/2, pos_means, width, 
            label=f"{cond} - ARPC3+", color='salmon' if i==0 else 'lightcoral')
    plt.bar(pos + i*width + width/2, neg_means, width, 
            label=f"{cond} - ARPC3-", color='skyblue' if i==0 else 'lightsteelblue')

plt.xlabel('Cell Region', fontweight='bold')
plt.ylabel('Mean CME Lifetime (s)', fontweight='bold')
plt.title('Mean CME Lifetimes by Region, Condition, and ARPC3 Status', fontweight='bold')
plt.xticks(pos + width/2, regions)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:

# Function to calculate percentages
def calculate_percentages(df):
    total_tracks = df['track_id'].nunique()
    c1_plus_c2_minus = df[(df['channel1_positive']) & (~df['channel2_positive'])]['track_id'].nunique() / total_tracks * 100
    c2_plus_c1_minus = df[(df['channel2_positive']) & (~df['channel1_positive'])]['track_id'].nunique() / total_tracks * 100
    double_positive = df[(df['channel1_positive']) & (df['channel2_positive'])]['track_id'].nunique() / total_tracks * 100
    
    return {'ARPC3+ DNM2-': c1_plus_c2_minus, 'DNM2+ ARPC3-': c2_plus_c1_minus, 'ARPC3+ DNM2+': double_positive}

# Compute percentages for each condition
control_percentages = calculate_percentages(filtered_tracks_control)
treatment_percentages = calculate_percentages(filtered_tracks_treatment)

# Convert to DataFrame for plotting
data = []
for condition, percentages in zip([label1, label2], [control_percentages, treatment_percentages]):
    for category, value in percentages.items():
        data.append({'Condition': condition, 'Category': category, 'Percentage': value})

df_plot = pd.DataFrame(data)

# Plot using seaborn barplot
plt.figure(figsize=(10, 6))
sns.barplot(x='Category', y='Percentage', hue='Condition', data=df_plot, palette={label1: 'orange', label2: 'blue'}, capsize=0.1)

# Customize plot
plt.xlabel("Category")
plt.ylabel("Percentage of Tracks")
plt.title("Percentage Distribution of Track Categories")
plt.legend(title="Condition")
plt.ylim(0, df_plot['Percentage'].max() + 10)  # Adjust y-axis for better visualization
plt.show()


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Function to calculate percentage of C1+ tracks
def calculate_c1_positive_percentage(df):
    total_tracks = df['track_id'].nunique()
    c1_positive = df[df['channel1_positive']]['track_id'].nunique() / total_tracks * 100
    return c1_positive

# Compute C1+ percentages for each condition
control_c1_percentage = calculate_c1_positive_percentage(filtered_tracks_control)
treatment_c1_percentage = calculate_c1_positive_percentage(filtered_tracks_treatment)

# Create DataFrame for plotting
df_plot = pd.DataFrame({
    'Condition': [label1, label2],
    'Percentage': [control_c1_percentage, treatment_c1_percentage]
})

# Set color mapping
palette_colors = {label1: 'orange', label2: 'blue'}

# Plot using seaborn barplot
plt.figure(figsize=(6, 6))
sns.barplot(x='Condition', y='Percentage', hue='Condition', data=df_plot, palette=palette_colors, capsize=0.1, legend=False)

# Customize plot
plt.xlabel("Condition")
plt.ylabel("Percentage of ARPC3+ Tracks")
plt.title("Percentage of ARPC3+ Tracks in" + ' ' + label1 + ' ' + 'vs' + ' ' + label2)
plt.ylim(0, df_plot['Percentage'].max() + 10)  # Adjust y-axis for better visualization
plt.show()



FOR THE PARAMETER SWEEP

In [ ]:
# # Define the base directory
# base_dir = r'C:\Users\Lab admin\Desktop\u-track3D\testTrackability\OS_41_5-10min-01_processed-cropped_channel3_analysis\tracking_results'

# # Create a dictionary to store all tracks
# filtered_tracks = {}

# # Loop through the numbers 1 to 12
# for i in range(1, 13):
#     # Construct the file path
#     file_path = os.path.join(base_dir, str(i), f'filtered_tracks_final_{i}.pkl')
    
#     # Load the pickle file and store it in the dictionary
#     filtered_tracks[i] = pd.read_pickle(file_path)

# # Now you can access the tracks like filtered_tracks[1], filtered_tracks[2], etc.


In [ ]:
# # range of track lengths to plot in each cohort, in frames
# custom_length_ranges = [[5, 10], [11,15], [16, 20], [21, 25], [26, 30], [31, 40], [41,90]]

In [ ]:
# plot_all_channels = False
# tracks_channel_subset = {}

# if plot_all_channels:
#     for i in range(1, 13):
#         tracks_channel_subset[i] = filtered_tracks[i].copy()

# else:
#     channels_to_plot = [True, True]

#     for i in range(1, 13):
#         tracks_channel_subset[i] = filtered_tracks[i][(filtered_tracks[i]['channel1_positive'] == channels_to_plot[0]) &
#                                     (filtered_tracks[i]['channel2_positive'] == channels_to_plot[1])]

In [ ]:
# # Plot histogram of all track lengths (lifetimes)
# plt.figure(figsize=(10, 6))
# colors = ['orange', 'blue', 'green', 'purple', 'red', 'pink']

# all_lifetimes = {}
# for i in range(1, 13):
#     all_lifetimes[i] = tracks_channel_subset[i]['track_length']*framerate_msec/1000


# bins = np.linspace(0, 150, 51)  # Define 50 bins between 0 and 150
# for i in range(1, 3):
#     plt.hist(all_lifetimes[i], bins=bins, color=colors[i-1], edgecolor='black', alpha=0.5, label=f'Track {i}')



# plt.xlabel('Time (s)', fontsize = 12, weight = 'bold')  # X-axis label
# plt.ylabel('Frequency', fontsize = 12, weight = 'bold')  # Y-axis label
# # plt.title('Histogram of AP2-TagRFP-T lifetimes (s) - Isotonic', fontsize = 14, weight = 'bold')  # Title of the histogram
# plt.title('Histogram of AP2-TagRFP-T lifetimes', fontsize = 14, weight = 'bold')  # Title of the histogram

# plt.legend(prop = {'weight':'bold'})
# plt.show()

In [ ]:
# import matplotlib.pyplot as plt
# import numpy as np

# # Increase font size for all text elements
# plt.rcParams.update({'font.size': 14})  # Set base font size larger

# # Define parameters from the key
# params = {
#     1: {"Merge/Split": "no", "minSearchRadius": 1, "timeWindow": 1},
#     2: {"Merge/Split": "no", "minSearchRadius": 1, "timeWindow": 2},
#     3: {"Merge/Split": "no", "minSearchRadius": 1, "timeWindow": 3},
#     4: {"Merge/Split": "no", "minSearchRadius": 2, "timeWindow": 1},
#     5: {"Merge/Split": "no", "minSearchRadius": 2, "timeWindow": 2},
#     6: {"Merge/Split": "no", "minSearchRadius": 2, "timeWindow": 3},
#     7: {"Merge/Split": "yes", "minSearchRadius": 1, "timeWindow": 1},
#     8: {"Merge/Split": "yes", "minSearchRadius": 1, "timeWindow": 2},
#     9: {"Merge/Split": "yes", "minSearchRadius": 1, "timeWindow": 3},
#     10: {"Merge/Split": "yes", "minSearchRadius": 2, "timeWindow": 1},
#     11: {"Merge/Split": "yes", "minSearchRadius": 2, "timeWindow": 2},
#     12: {"Merge/Split": "yes", "minSearchRadius": 2, "timeWindow": 3}
# }

# # Define colors for each dataset
# colors = {
#     1: 'orange',
#     2: 'blue',
#     3: 'green',
#     4: 'purple',
#     5: 'red',
#     6: 'pink',
#     7: 'brown',
#     8: 'cyan',
#     9: 'magenta',
#     10: 'lime',
#     11: 'gray',
#     12: 'olive'
# }

# # Create a list of all datasets you have
# # Replace with actual datasets you have
# datasets = {
#     1: all_lifetimes[1],
#     2: all_lifetimes[2],
#     3: all_lifetimes[3],
#     # Uncomment and add more datasets as needed
#     4: all_lifetimes[4],
#     5: all_lifetimes[5],
#     6: all_lifetimes[6],
#     7: all_lifetimes[7],
#     8: all_lifetimes[8],
#     9: all_lifetimes[9],
#     10: all_lifetimes[10],
#     11: all_lifetimes[11],
#     12: all_lifetimes[12]
# }

# bins = np.linspace(0, 150, 51)  # Define 50 bins between 0 and 150

# # Define single-parameter change comparisons
# # Format: ("Parameter Name", [(dataset1, dataset2), ...])
# single_param_comparisons = [
#     # Effect of changing timeWindow (keeping other parameters constant)
#     ("Effect of timeWindow", [
#         (1, 2), (2, 3),  # no, radius=1, window: 1→2→3
#         (4, 5), (5, 6),  # no, radius=2, window: 1→2→3
#         (7, 8), (8, 9),  # yes, radius=1, window: 1→2→3
#         (10, 11), (11, 12)  # yes, radius=2, window: 1→2→3
#     ]),
    
#     # Effect of changing minSearchRadius (keeping other parameters constant)
#     ("Effect of minSearchRadius", [
#         (1, 4),  # no, window=1, radius: 1→2
#         (2, 5),  # no, window=2, radius: 1→2
#         (3, 6),  # no, window=3, radius: 1→2
#         (7, 10),  # yes, window=1, radius: 1→2
#         (8, 11),  # yes, window=2, radius: 1→2
#         (9, 12)   # yes, window=3, radius: 1→2
#     ]),
    
#     # Effect of changing Merge/Split (keeping other parameters constant)
#     ("Effect of Merge/Split", [
#         (1, 7),   # radius=1, window=1, Merge/Split: no→yes
#         (2, 8),   # radius=1, window=2, Merge/Split: no→yes
#         (3, 9),   # radius=1, window=3, Merge/Split: no→yes
#         (4, 10),  # radius=2, window=1, Merge/Split: no→yes
#         (5, 11),  # radius=2, window=2, Merge/Split: no→yes
#         (6, 12)   # radius=2, window=3, Merge/Split: no→yes
#     ])
# ]

# # Create figure with subplots for each parameter type
# n_rows = len(single_param_comparisons)
# max_cols = max(len(comps[1]) for comps in single_param_comparisons)

# fig = plt.figure(figsize=(max_cols * 9, n_rows * 10))
# gs = fig.add_gridspec(n_rows, 1, height_ratios=[1] * n_rows)

# for row, (param_name, comparisons) in enumerate(single_param_comparisons):
#     # Create subgridspec for this parameter's comparisons
#     subgs = gs[row].subgridspec(1, max_cols)
    
#     # Add a row title (parameter being changed)
#     row_title = fig.add_subplot(gs[row])
#     row_title.set_title(param_name, fontsize=24, fontweight='bold')
#     row_title.set_frame_on(False)
#     row_title.set_xticks([])
#     row_title.set_yticks([])
    
#     # Plot each comparison in this row
#     for col, (dataset1, dataset2) in enumerate(comparisons):
#         # Skip if either dataset doesn't exist
#         if dataset1 not in datasets or dataset2 not in datasets:
#             continue
            
#         # Create subplot
#         ax = fig.add_subplot(subgs[0, col])
        
#         # Plot histograms
#         ax.hist(datasets[dataset1], bins=bins, color=colors[dataset1], 
#                 edgecolor='black', alpha=0.5, 
#                 label=f'{dataset1}: {params[dataset1]["Merge/Split"]}, '
#                      f'radius={params[dataset1]["minSearchRadius"]}, '
#                      f'window={params[dataset1]["timeWindow"]}')
        
#         ax.hist(datasets[dataset2], bins=bins, color=colors[dataset2], 
#                 edgecolor='black', alpha=0.35, 
#                 label=f'{dataset2}: {params[dataset2]["Merge/Split"]}, '
#                      f'radius={params[dataset2]["minSearchRadius"]}, '
#                      f'window={params[dataset2]["timeWindow"]}')
        
#         # Determine which parameter is changing
#         changing_param = ""
#         if params[dataset1]["Merge/Split"] != params[dataset2]["Merge/Split"]:
#             changing_param = "Merge/Split"
#         elif params[dataset1]["minSearchRadius"] != params[dataset2]["minSearchRadius"]:
#             changing_param = "radius"
#         elif params[dataset1]["timeWindow"] != params[dataset2]["timeWindow"]:
#             changing_param = "window"
            
#         # Set title and labels
#         ax.set_title(f'Change {changing_param}: {dataset1} vs {dataset2}', fontsize = 18)
#         ax.set_xlabel('Lifetime', fontsize = 16)
#         ax.set_ylabel('Frequency', fontsize = 16)
#         ax.legend(loc='upper right', fontsize=14, framealpha=0.9)

# # Remove the row title axes from the layout calculations
# plt.tight_layout(rect=[0, 0, 1, 0.96], h_pad = 3.0, w_pad = 3.0)  # Adjust top margin for row titles
# plt.suptitle('Effect of Individual Parameter Changes', fontsize=28, y=0.99)


# # # Save as a png with dpi 600 and bbox_inches='tight' in C:\Users\Lab admin\Desktop\u-track3D\testTrackability\OS_41_5-10min-01_processed-cropped_channel3_analysis\tracking_results
# # plt.savefig(r'C:\Users\Lab admin\Desktop\u-track3D\testTrackability\OS_41_5-10min-01_processed-cropped_channel3_analysis\tracking_results\Effect_of_Individual_Parameter_Changes.png', dpi=600, bbox_inches='tight')
# # plt.close(fig)  # Close the figure to free memory

# plt.show()

In [ ]:
# # Increase font size for all text elements
# plt.rcParams.update({'font.size': 14})  # Set base font size larger

# # Modify these lines in your plotting code:
# fig = plt.figure(figsize=(max_cols * 9, n_rows * 10))  # Make figure larger

# # Increase title font sizes
# row_title.set_title(param_name, fontsize=24, fontweight='bold')
# ax.set_title(f'Change {changing_param}: {dataset1} vs {dataset2}', fontsize=18)
# plt.suptitle('Effect of Individual Parameter Changes', fontsize=28, y=0.99)

# # Increase label font sizes
# ax.set_xlabel('Lifetime', fontsize=16)
# ax.set_ylabel('Frequency', fontsize=16)

# # Improve legend formatting
# ax.legend(loc='upper right', fontsize=14, framealpha=0.9)  # Make legend more visible

# # Add more space between subplots
# plt.tight_layout(rect=[0, 0, 1, 0.96], h_pad=3.0, w_pad=3.0)

In [ ]:
# gs = fig.add_gridspec(n_rows, 1, height_ratios=[1] * n_rows, hspace=0.4)
# subgs = gs[row].subgridspec(1, max_cols, wspace=0.3)

In [ ]:
# # Intensities of detected spots in each channel
# cleaned_tracks_control = pd.read_pickle('/Users/matsulab/Desktop/LLSM-CME-ANALYSIS/Final/movie_data/datasets/track_df_cleaned_final_full.pkl')
# cleaned_tracks_ck666 = pd.read_pickle('/Users/matsulab/Desktop/LLSM-CME-ANALYSIS/Final/movie_data_CK66650uM_5-10min/datasets/track_df_cleaned_final_full.pkl')
# # cleaned_tracks_ck666 = pd.read_pickle('/Users/matsulab/Desktop/LLSM-CME-ANALYSIS/Final/movie_data_CK66650uM_5-10min/datasets/track_df_cleaned_final_full.pkl')

In [ ]:
# # plot intensity histograms for fun
# # change the 'c3_voxel_sum_adjusted' value to plot other intensity metrics
# cleaned_tracks_control_long = cleaned_tracks_control[cleaned_tracks_control['c1_voxel_sum_adjusted'] > 2000]
# cleaned_tracks_ck666_long = cleaned_tracks_ck666[cleaned_tracks_ck666['c1_voxel_sum_adjusted'] > 2000]

# # plt.hist(cleaned_tracks_ck666['c3_voxel_sum_adjusted'], bins=100, alpha=0.5, label='Channel 3')
# # plt.hist(cleaned_tracks_ck666['c2_voxel_sum_adjusted'], bins=100, alpha=0.5, label='Channel 2')
# plt.hist(cleaned_tracks_ck666_long['c1_voxel_sum_adjusted'], bins=100, alpha=0.5, label='Channel 1 ck666')

# # plt.hist(cleaned_tracks_control['c3_voxel_sum_adjusted'], bins=100, alpha=0.5, label='Channel 3')
# # plt.hist(cleaned_tracks_control['c2_voxel_sum_adjusted'], bins=100, alpha=0.5, label='Channel 2')
# plt.hist(cleaned_tracks_control_long['c1_voxel_sum_adjusted'], bins=100, alpha=0.5, label='Channel 1 control')

# # histogram of three channels with uniform bin sizes
# # plt.hist(track_df['c3_voxel_sum_adjusted'], bins=np.linspace(-5000, 10000, 100), alpha=0.5, label='Channel 3')
# # plt.hist(track_df['c2_voxel_sum_adjusted'], bins=np.linspace(-5000, 10000, 100), alpha=0.5, label='Channel 2')
# # plt.hist(track_df['c1_voxel_sum_adjusted'], bins=np.linspace(-5000, 10000, 100), alpha=0.5, label='Channel 1')

# plt.legend()
# plt.title('Voxel sum adjusted')
# plt.xlabel('Voxel sum')
# plt.ylabel('Count')
# plt.xticks([-5000, -2500, 0, 2500, 5000, 7500, 10000, 12500, 15000])
# plt.xlim(-5000, 20000)
# # plt.xlim(-5000, 10000)
# plt.show()

In [ ]:
# Plot boxplots of channel 1 on the same plot (is CK666 higher than control?)
# Make sure to compare the histograms to the outputs from notebook 4

In [ ]:
# import pandas as pd
# import seaborn as sns
# import matplotlib.pyplot as plt

# # Assuming you already have cleaned_tracks_control and cleaned_tracks_ck666 dataframes
# # Creating a combined dataframe with a 'Tag' column to differentiate between control and ck666
# control_data = cleaned_tracks_control[['c1_voxel_sum_adjusted', 'c2_voxel_sum_adjusted', 'c3_voxel_sum_adjusted']].copy()
# control_data['Tag'] = 'Control'

# ck666_data = cleaned_tracks_ck666[['c1_voxel_sum_adjusted', 'c2_voxel_sum_adjusted', 'c3_voxel_sum_adjusted']].copy()
# ck666_data['Tag'] = 'CK666'

# # Combine both dataframes into one
# combined_data = pd.concat([control_data, ck666_data])

# # Reshape the dataframe to make it easier for seaborn boxplot
# combined_data_melted = pd.melt(combined_data, id_vars=['Tag'], value_vars=['c1_voxel_sum_adjusted', 'c2_voxel_sum_adjusted', 'c3_voxel_sum_adjusted'],
#                                var_name='Channel', value_name='Intensity')

# # Map channel names to the desired labels
# channel_labels = {'c1_voxel_sum_adjusted': 'ARPC3', 'c2_voxel_sum_adjusted': 'DNM2', 'c3_voxel_sum_adjusted': 'AP2'}
# combined_data_melted['Channel'] = combined_data_melted['Channel'].map(channel_labels)

# # Plot the boxplot
# plt.figure(figsize=(10, 6))
# sns.set(style="ticks")  # Remove gridlines

# # Creating the boxplot
# sns.boxplot(x='Channel', y='Intensity', hue='Tag', data=combined_data_melted, palette={'Control': 'orange', 'CK666': 'blue'}, dodge=True)

# # Set y-axis limit
# plt.ylim(-15000, 40000)

# # Adding labels and title with bold font
# plt.xlabel('Tagged Protein', fontsize=12, fontweight='bold')
# plt.ylabel('Intensity', fontsize=12, fontweight='bold')
# plt.title('Intensity Distribution for Each Tagged Protein', fontsize=14, fontweight='bold')
# plt.legend(title='Condition', loc='upper right', fontsize=10, title_fontsize='12', frameon=False)

# # Show the plot
# plt.tight_layout()
# plt.show()
